# Akıl Hocası — ilk eğitim ve kapasite artırımı (isteğe bağlı hızlandırıcı)

Bu not defteri **bir kez** (veya büyük sıçramalarda) kullanılır; sonrasında program her gece kendi kendini
eğitir (öğrenen sıralayıcı, örnek hafıza, terim öğrenme ve donanım yeterliyse yerel ek katman eğitimi).

**Kural:** Model mevzuat ezberlemez. Eğitim verisi, programın doğrulama motorundan geçmiş ve personelin
"faydalı" işaretlediği cevaplardır; model *verilen kaynak parçalarından doğrulanabilir cevap yazmayı* öğrenir.

**Adımlar**
1. Programda: Neler Değişti › Öğrenme › **Eğitim verisini dışa aktar** → `egitim.jsonl`
2. Kaggle'da: Datasets › New Dataset › `egitim.jsonl` yükleyin (ad: `akilhocasi-egitim`)
3. Bu not defterini Kaggle'a yükleyin, sağ panelden **GPU T4 x2** (veya P100) ve **Internet: On** seçin, Run All
4. Çıkan `.gguf` dosyasını indirip programda Ayarlar › Yapay zekâ modeli › **Aday modeli dene** ile seçin.
   Program yeni modeli altın soru setinde eskisiyle karşılaştırır; **yalnızca daha başarılıysa** devreye alır.

> Not: Kaggle'daki kütüphane sürümleri değiştikçe kurulum hücresinde küçük düzeltme gerekebilir.

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
# Ayarlar — temel model Faz 0'daki Türkçe testine göre seçilir
TEMEL_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # alternatif: daha küçük/büyük Qwen veya Gemma modelleri
VERI = "/kaggle/input/akilhocasi-egitim/egitim.jsonl"
MAKS_UZUNLUK = 4096
TUR = 2            # epoch
CIKTI = "akilhocasi-model"

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(TEMEL_MODEL, max_seq_length=MAKS_UZUNLUK, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth")

In [ ]:
from datasets import load_dataset
veri = load_dataset("json", data_files=VERI, split="train")
veri = veri.map(lambda o: {"text": tokenizer.apply_chat_template(o["messages"], tokenize=False)})
bolum = veri.train_test_split(test_size=0.1, seed=7) if len(veri) >= 50 else {"train": veri, "test": None}
print(len(veri), "doğrulanmış örnek")

In [ ]:
from trl import SFTTrainer, SFTConfig
egitici = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=bolum["train"], eval_dataset=bolum["test"],
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAKS_UZUNLUK, per_device_train_batch_size=2,
                   gradient_accumulation_steps=4, num_train_epochs=TUR, learning_rate=2e-4, warmup_ratio=0.05,
                   logging_steps=10, optim="adamw_8bit", output_dir="egitim_cikti", report_to="none"))
egitici.train()

In [ ]:
# Programın kullandığı biçim: GGUF, 4 bit (Q4_K_M)
model.save_pretrained_gguf(CIKTI, tokenizer, quantization_method="q4_k_m")
import glob; print(glob.glob(f"{CIKTI}*/*.gguf") + glob.glob(f"{CIKTI}*.gguf"))

## Sonraki adım
`.gguf` dosyasını indirin (sağ panel › Output). Programda **Aday modeli dene** ile seçin.
Karşılaştırma sonucu ekranda görünür: yeni model altın sette daha fazla soruyu doğru cevaplar ve kritik hatası
artmazsa devreye girer; aksi halde eski model kalır ve aday dosya `modeller\aday` klasöründe bekler.